# 1 — Data availability inventory

**Which households can support any analysis at all?**

Source: `recon.py`. Reads only metadata and overview files — never the 5.26 GB of
time series.

> ⚠️ **The outputs below are already saved — just scroll and read.**
> Do **not** press *Run*. This reads the 5 GB HEAPO dataset from a local folder
> that is not attached here, so re-running produces only `FileNotFoundError`.
> Everything you need to see is stored in the cells.

### The four questions
1. How many households have a separate heat pump meter?
2. How long is each household's history?
3. How many complete heating seasons does each span?
4. Among visited households, how many have data on both sides of the visit?


In [1]:
import recon

frames = {key: recon.load(key) for key in recon.FILES}
for name, df in frames.items():
    print(f"{name:12s} {df.shape[0]:>5} rows x {df.shape[1]:>3} cols")

households    1408 rows x  11 cols
daily         1298 rows x  13 cols
15min         1407 rows x   9 cols
protocols      410 rows x 106 cols


## Step 1 — Inspect before assuming

Dtypes, samples, and the distinct values of every low-cardinality column. This is
where the column-naming inconsistency in the daily overview shows up.

In [2]:
recon.inspect(frames)

STEP 1 -- STRUCTURE INSPECTION

--- households: households.csv  shape=(1408, 11) ---
dtypes:
Household_ID                         int64
Group                                  str
Weather_ID                             str
Installation_HasPVSystem            object
Protocols_Available                   bool
Protocols_HasMultipleVisits           bool
Protocols_ReportIDs                    str
MetaData_Available                    bool
SmartMeterData_Available_15min        bool
SmartMeterData_Available_Daily        bool
SmartMeterData_Available_Monthly      bool
head(3):
   Household_ID      Group Weather_ID Installation_HasPVSystem  Protocols_Available  Protocols_HasMultipleVisits Protocols_ReportIDs  MetaData_Available  SmartMeterData_Available_15min  SmartMeterData_Available_Daily  SmartMeterData_Available_Monthly
0        661091  treatment        MqO                    False                 True                        False                [29]                True                      

## Build the merged inventory

Left joins keep all 1,408 households. A household absent from an overview file gets
NaN there — and that NaN is itself the answer to "is this data available?".

In [3]:
inv = recon.build_inventory(frames)
print(f"merged inventory: {inv.shape[0]} rows x {inv.shape[1]} columns")

# Same banner recon.main() emits, so this notebook's transcript matches the
# verified outputs/recon_summary.txt line for line.
recon.say("=" * 78)
recon.say("STEP 2 -- ANSWERS")
recon.say("=" * 78)
recon.say("")

merged inventory: 1408 rows x 37 columns
STEP 2 -- ANSWERS



## Q1 — Sub-metering

In [4]:
recon.q1_submetering(inv)

Q1 -- SUB-METERING (separate heat pump meter)
MeasurementsAvailable_HeatPump is a boolean flag: True = a dedicated heat
pump channel exists for that household. NaN = household absent from the file.

[daily overview]  households present in file: 1298 / 1408
  with heat pump sub-meter: 59
HasHeatPumpMeter  False  True 
Group                         
control            1145     49
treatment           204     10

in overview]  households present in file: 1407 / 1408
  with heat pump sub-meter: 93
HasHeatPumpMeter  False  True 
Group                         
control            1134     60
treatment           181     33

Agreement on the 1298 households present in BOTH files:
15min  False  True 
daily              
False   1239      0
True       0     59
  disagreements: 0
  sub-metered in 15min but ABSENT from the daily file: 34



## Q2 — History length

In [5]:
recon.q2_history(inv)

Q2 -- HISTORY LENGTH (SMD_daily_TimeAvailable_NumberDays)
count    1298.000000
mean      721.129430
std       432.583995
min         2.000000
25%       432.250000
50%       698.500000
75%       917.750000
max      1966.000000

  >=  365 days:  1041 households (80.2% of the 1298 with daily data)
  >=  730 days:   606 households (46.7% of the 1298 with daily data)

15min overview, for comparison:
  n=1407  mean=638.7  median=559  min=20  max=1675
  >=365d: 1254   >=730d: 372



## Q3 — Winter coverage

In [6]:
recon.q3_winters(inv)

Q3 -- WINTER COVERAGE (complete heating seasons, 1 Oct - 31 Mar)
*** UPPER BOUND: counted from the earliest/latest timestamps only. A season
*** counts when it lies entirely inside the household's date range. This does
*** NOT account for gaps or missing days within that range -- the true number
*** of usable winters can only be lower, never higher.

[daily overview]  n=1298
  0 complete winter(s):   379 households
  1 complete winter(s):   556 households
  2 complete winter(s):   228 households
  3 complete winter(s):    51 households
  4 complete winter(s):    84 households
  mean=1.16  median=1  max=4
  >= 1 complete winter : 919
  >= 2 complete winters: 363

in overview]  n=1407
  0 complete winter(s):   562 households
  1 complete winter(s):   542 households
  2 complete winter(s):   204 households
  3 complete winter(s):    98 households
  4 complete winter(s):     1 households
  mean=0.89  median=1  max=4
  >= 1 complete winter : 845
  >= 2 complete winters: 303



## Q4 — Before/after balance

In [7]:
recon.q4_balance(inv, frames['protocols'])

Q4 -- BEFORE/AFTER BALANCE (households with protocol data)
protocols.csv: 410 visit rows, 214 distinct households (193 rows carry no Household_ID)
households with Protocols_Available=True: 214 (exactly the Group=='treatment' set)

[daily overview]  base: 156 protocol households with daily data
 threshold_days  before>=t  after>=t  BOTH>=t pct_both
             90        115        60       40    25.6%
            180         99        36       20    12.8%
            365         73        31       11     7.1%

in overview]  base: 214 protocol households with 15min data
 threshold_days  before>=t  after>=t  BOTH>=t pct_both
             90        146       167      109    50.9%
            180        131       157       89    41.6%
            365         98       139       68    31.8%



## What this establishes

- **59** households have heat pump sub-metering in the daily archive, **93** at 15min.
  The two agree perfectly where both files cover the same household.
- Median history is **698 days**; 1,041 households have ≥1 year.
- **89** visited households have ≥180 days of 15-minute data on *both* sides of the
  visit. The same rule on daily files yields only **20** — the daily archive largely
  stops at the visit date.

That **89** is the base sample for everything that follows.
